[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Xrenya/stable_diffusion/blob/main/SD3.ipynb)

# Stable Diffusion 3 — MMDiT Transformer from Scratch

This tutorial builds the **MMDiT (Multimodal Diffusion Transformer)** — the
core model of **Stable Diffusion 3.x** — from scratch in plain PyTorch.

SD3 represents a **generational leap**: it replaces the U-Net entirely with a
pure Transformer architecture (DiT-style), fundamentally changing how image
generation works.

## What Changed from SD 1.x/2.x to SD 3.x?

| Feature | SD 1.x / 2.x | SD 3.x |
|---------|--------------|--------|
| Core architecture | **UNet** (conv + attention) | **Transformer** (DiT) |
| Image representation | Conv feature maps | **Patches** (like ViT) |
| Text conditioning | Cross-attention (text queries image) | **Joint attention** (MMDiT) |
| Normalization | GroupNorm | **AdaLN-Zero** + RMSNorm |
| Latent channels | 4 | **16** |
| Text encoders | 1 CLIP (SD1) or 1 OpenCLIP (SD2) | **3**: CLIP-L + OpenCLIP-bigG + T5-XXL |
| Noise schedule | DDPM (predict noise ε) | **Rectified Flow** (predict velocity v) |
| Attention Q/K norm | None | **RMSNorm** (per-head) |

## The Key Innovation: Joint Attention (MMDiT)

In SD 1.x/2.x, text and image interact via **cross-attention**: image features
query text embeddings (asymmetric). In SD3, image and text tokens are
**concatenated into a single sequence** and attend to each other jointly
(symmetric). Each modality keeps its own FFN, but attention is shared:

```
SD 1.x/2.x (UNet):                        SD 3.x (MMDiT):
----------------------------------     --------------------------
Image => Self-Attn => Cross-Attn => FFN   Image tokens + Text tokens
               ^                                    |
          Text (K,V only)                      Joint Self-Attn
                                                    |
                                            Split back apart
                                           ┌--------|-------┐
                                        Image FFN     Text FFN
```

## What This Tutorial Covers

This notebook implements the **MMDiT transformer only** (the diffusion backbone).
A full SD3 pipeline would also require the VAE (16-channel), three text encoders,
and a rectified flow sampler.

| Component | Purpose |
|---|---|
| `PatchEmbed` | Latent (16, H, W) => patch tokens (ViT-style) |
| `RMSNorm` | Q/K normalization for training stability |
| `AdaLayerNormZero` | Timestep conditioning via shift/scale/gate |
| `JointAttention` | The core MMDiT mechanism |
| `JointTransformerBlock` | Dual-stream block (image + text) |
| `SD3Transformer2DModel` | Full model: 24 blocks, ~2B params |

## 0. Dependencies

In [ ]:
!pip install torch safetensors huggingface_hub diffusers==0.29.2 transformers==4.44.2

## 1. Configuration

Like SD 2.x, SD3 uses a config-driven architecture. Key numbers for
**SD3-Medium**:

- **24 transformer blocks**, each with 24 attention heads × 64 head dim = **1536 inner dim**
- **16 latent channels** (SD 1.x/2.x used 4) with patch_size=2
- **joint_attention_dim = 4096** from T5-XXL text encoder
- **pooled_projection_dim = 2048** from CLIP-L (768) + OpenCLIP-bigG (1280)
- **pos_embed_max_size = 192** supports up to ~1536×1536 pixel images

In [ ]:
import math
import json
import os
from dataclasses import dataclass, field
from typing import Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file


@dataclass
class SD3TransformerConfig:
    """
    Configuration for the SD3 MMDiT transformer.
    Defaults are for SD3-Medium.
    """
    sample_size: int = 128              # Latent spatial size (image_size / 8)
    patch_size: int = 2                 # ViT-style patch size
    in_channels: int = 16               # SD3 VAE has 16 latent channels (vs 4 in SD 1.x)
    out_channels: int = 16  
    num_layers: int = 24                # Number of MMDiT blocks
    num_attention_heads: int = 24       # Attention heads per block
    attention_head_dim: int = 64        # Dimension per head
    joint_attention_dim: int = 4096     # T5-XXL hidden dim
    caption_projection_dim: int = 1536  # = inner_dim (24 * 64)
    pooled_projection_dim: int = 2048   # CLIP-L (768) + OpenCLIP-bigG (1280)
    pos_embed_max_size: int = 192       # Max positional embedding grid

## 2. Positional Embedding (2D Sinusoidal)

Since SD3 treats the latent as patches (like ViT), it needs **2D positional
embeddings** to encode spatial location. Each patch gets an embedding that
encodes its (row, column) position.

The embedding is split in half: first half encodes the **width** axis,
second half encodes the **height** axis. Each half uses sinusoidal encoding
(same formula as the original Transformer).

The embeddings are precomputed for a **maximum grid size** (192×192) and
**cropped** at runtime to the actual size. This allows variable-resolution
inference without recomputing.

In [ ]:
def get_1d_sincos_pos_embed(embed_dim: int, pos: torch.Tensor) -> torch.Tensor:
    """
    Standard 1D sinusoidal positional embedding.

    pos: (N,) position indices
    returns: (N, embed_dim)
    """
    half = embed_dim // 2
    omega = 1.0 / (10000.0 ** (torch.arange(half, dtype=torch.float32, device=pos.device) / half))
    out = pos.float()[:, None] * omega[None, :]     # (N, half)
    emb = torch.cat([torch.sin(out), torch.cos(out)], dim=-1)  # (N, embed_dim)
    return emb


def get_2d_sincos_pos_embed(
    embed_dim: int,
    grid_size: int,
    base_size: int = 64,
    interpolation_scale: float = 1.0,
) -> torch.Tensor:
    """
    Compute 2D sinusoidal positional embedding for a grid of patches.

    First half of embed_dim encodes the width axis.
    Second half encodes the height axis.

    Returns: (grid_size^2, embed_dim)
    """
    # Create scaled position grids
    pos_h = torch.arange(grid_size, dtype=torch.float32) / (grid_size / base_size) / interpolation_scale
    pos_w = torch.arange(grid_size, dtype=torch.float32) / (grid_size / base_size) / interpolation_scale
    grid_h, grid_w = torch.meshgrid(pos_h, pos_w, indexing="ij")
    grid_h = grid_h.reshape(-1)
    grid_w = grid_w.reshape(-1)

    # Encode each axis with half the embedding dimension
    half = embed_dim // 2
    emb_h = get_1d_sincos_pos_embed(half, grid_h)
    emb_w = get_1d_sincos_pos_embed(half, grid_w)
    return torch.cat([emb_w, emb_h], dim=-1)  # (grid_size^2, embed_dim)

## 3. RMSNorm (QK Normalization)

SD3 applies **RMSNorm** to queries and keys before computing attention. This
prevents attention logits from growing too large during training, which is
critical for training stability at scale.

Unlike LayerNorm, RMSNorm does **not** center the input (no mean subtraction).
It only normalizes by the root mean square, then applies a learned scale:

$$\text{RMSNorm}(x) = \frac{x}{\sqrt{\text{mean}(x^2) + \epsilon}} \cdot \gamma$$

The normalization is applied **per attention head** (dim = head_dim = 64),
not over the full embedding. SD 1.x/2.x had no QK normalization at all.

In [ ]:
class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization (no centering, just scaling)."""

    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))  # Learned scale

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (..., dim) - applied per-head in attention
        rms = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x * rms * self.weight

## 4. Timestep Embedding

Same sinusoidal approach as SD 1.x/2.x, but with a key difference:
in **rectified flow**, timesteps are **continuous floats in [0, 1]** rather
than integer step indices (0–1000). The sinusoidal embedding handles both.

The 256-dim sinusoidal encoding is projected through a 2-layer MLP to the
model's inner dimension (1536).

In [ ]:
def sinusoidal_timestep_embedding(
    timesteps: torch.Tensor,
    dim: int,
    flip_sin_to_cos: bool = True,
    downscale_freq_shift: float = 0.0,
    max_period: int = 10000,
) -> torch.Tensor:
    """
    Sinusoidal timestep embedding.

    timesteps: (B,) float in [0, 1] for rectified flow
    returns: (B, dim)
    """
    half = dim // 2
    exponent = -math.log(max_period) * torch.arange(
        half, dtype=torch.float32, device=timesteps.device
    )
    exponent = exponent / (half - downscale_freq_shift)
    freqs = torch.exp(exponent)

    args = timesteps.float()[:, None] * freqs[None, :]
    emb = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)

    if flip_sin_to_cos:
        emb = torch.cat([emb[:, half:], emb[:, :half]], dim=-1)

    if dim % 2 == 1:
        emb = F.pad(emb, (0, 1))
    return emb


class TimestepEmbedding(nn.Module):
    """Two-layer MLP: sinusoidal features (256) -> inner_dim (1536)."""

    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.linear_1 = nn.Linear(in_channels, out_channels, bias=True)
        self.act = nn.SiLU()
        self.linear_2 = nn.Linear(out_channels, out_channels, bias=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear_2(self.act(self.linear_1(x)))

## 5. Pooled Text Projection

SD3 uses **three text encoders**, but their outputs serve different purposes:

- **T5-XXL** (4096-dim): Provides the token-level sequence embeddings that
  participate in joint attention (projected to 1536-dim).

- **CLIP-L** (768-dim, pooled) + **OpenCLIP-bigG** (1280-dim, pooled):
  Their pooled [CLS] outputs are concatenated (768 + 1280 = **2048-dim**)
  and projected to drive the **AdaLN-Zero conditioning**.

The pooled embeddings capture "global semantics" of the prompt (what the
image is about), while the T5 sequence embeddings capture fine-grained
token-level information.

In [ ]:
class TextProjection(nn.Module):
    """
    Projects concatenated pooled CLIP-L + OpenCLIP-bigG outputs (2048-dim)
    to the model's inner dim (1536). Uses GELU(tanh) activation.
    """

    def __init__(self, in_features: int, hidden_size: int):
        super().__init__()
        self.linear_1 = nn.Linear(in_features, hidden_size, bias=True)
        self.act_1 = nn.GELU(approximate="tanh")
        self.linear_2 = nn.Linear(hidden_size, hidden_size, bias=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear_2(self.act_1(self.linear_1(x)))

## 6. Combined Timestep + Pooled Text Embedding

The conditioning signal for AdaLN-Zero is the **sum** of:
1. Projected sinusoidal timestep embedding
2. Projected pooled text embedding

This single vector (1536-dim) drives all the shift/scale/gate parameters
in every transformer block. By combining time and text, the model knows both
"what noise level am I at" and "what should the image be about" in a single
conditioning signal.

In [ ]:
class CombinedTimestepTextProjEmbeddings(nn.Module):
    """
    Combines timestep + pooled text into a single conditioning vector.

    sinusoidal(timestep) -> MLP -> 1536-dim
    pooled_text (2048)   -> MLP -> 1536-dim
    Sum -> conditioning vector (1536-dim)
    """

    def __init__(self, embedding_dim: int, pooled_projection_dim: int):
        super().__init__()
        self.timestep_embedder = TimestepEmbedding(
            in_channels=256,            # Sinusoidal embedding dim
            out_channels=embedding_dim, # 1536
        )
        self.text_embedder = TextProjection(
            in_features=pooled_projection_dim,  # 2048
            hidden_size=embedding_dim,           # 1536
        )

    def forward(self, timestep: torch.Tensor, pooled_projection: torch.Tensor) -> torch.Tensor:
        # Compute sinusoidal features (no learnable params)
        timestep_emb = sinusoidal_timestep_embedding(
            timestep, dim=256, flip_sin_to_cos=True, downscale_freq_shift=0.0,
        )
        # Project both and sum
        timestep_emb = self.timestep_embedder(timestep_emb.to(dtype=pooled_projection.dtype))
        pooled_emb = self.text_embedder(pooled_projection)
        return timestep_emb + pooled_emb

## 7. Patch Embedding (ViT-style)

This is a fundamental architectural shift from SD 1.x/2.x. Instead of
processing the latent with convolutions (like a UNet), SD3 treats it
as a **sequence of patches** (like a Vision Transformer):

```
Latent: (B, 16, 128, 128)
  |
  |- Split into 2×2 patches: 64×64 = 4096 patches
  |- Each patch: 16 channels × 2 × 2 = 64 values
  |- Project each patch: 64 -> 1536 via Conv2d(k=2, s=2)
  |- Add 2D sinusoidal positional embedding
  |
  |- Output: (B, 4096, 1536) token sequence
```

The Conv2d with kernel=stride=patch_size is mathematically equivalent
to splitting into non-overlapping patches and applying a linear projection
to each patch.

Positional embeddings are precomputed for the max grid size and **cropped**
at runtime, enabling variable resolution without recomputation.

In [ ]:
class PatchEmbed(nn.Module):
    """
    Convert latent image into patch token sequence (ViT-style).

    Input:  (B, 16, H, W)
    Output: (B, num_patches, embed_dim)
    """

    def __init__(
        self,
        sample_size: int = 128,
        patch_size: int = 2,
        in_channels: int = 16,
        embed_dim: int = 1536,
        pos_embed_max_size: int = 192,
    ):
        super().__init__()
        self.patch_size = patch_size
        self.pos_embed_max_size = pos_embed_max_size
        self.base_size = sample_size // patch_size  # 64 for 128 sample, 2 patch

        # Conv2d with kernel=stride=patch_size acts as the patch projection
        # Each 2x2x16 patch -> 1 token of dim 1536
        self.proj = nn.Conv2d(
            in_channels, embed_dim,
            kernel_size=patch_size, stride=patch_size, bias=True,
        )

        # Precompute sinusoidal pos embed for the max grid
        # (not saved in checkpoint, recomputed on load)
        interpolation_scale = pos_embed_max_size / self.base_size
        pos_embed = get_2d_sincos_pos_embed(
            embed_dim,
            grid_size=pos_embed_max_size,
            base_size=self.base_size,
            interpolation_scale=interpolation_scale,
        )
        self.register_buffer(
            "pos_embed",
            pos_embed.unsqueeze(0),  # (1, max_size^2, embed_dim)
            persistent=False,
        )

    def cropped_pos_embed(self, height: int, width: int) -> torch.Tensor:
        """Crop the precomputed positional embedding to actual grid size."""
        top = (self.pos_embed_max_size - height) // 2
        left = (self.pos_embed_max_size - width) // 2
        spatial = self.pos_embed.reshape(
            1, self.pos_embed_max_size, self.pos_embed_max_size, -1
        )
        spatial = spatial[:, top : top + height, left : left + width, :]
        return spatial.reshape(1, height * width, -1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, 16, H, W)
        h = x.shape[2] // self.patch_size
        w = x.shape[3] // self.patch_size

        x = self.proj(x)                                  # (B, embed_dim, h, w)
        x = x.flatten(2).transpose(1, 2)                  # (B, h*w, embed_dim)
        x = x + self.cropped_pos_embed(h, w).to(x.dtype)  # Add position info
        return x

## 8. Adaptive Layer Normalization (AdaLN-Zero)

SD3 uses **AdaLN-Zero** instead of the simple GroupNorm + timestep addition
used in SD 1.x/2.x. This is a much more expressive conditioning mechanism.

### How it works
The conditioning vector (timestep + pooled text) is projected to produce
**6 modulation parameters** per token:

| Parameter | Applied to | Formula |
|---|---|---|
| `shift_msa` | Pre-attention LayerNorm | `LN(x) * (1 + scale) + shift` |
| `scale_msa` | Pre-attention LayerNorm | |
| `gate_msa` | Attention output | `x + gate * attn(x)` |
| `shift_mlp` | Pre-FFN LayerNorm | `LN(x) * (1 + scale) + shift` |
| `scale_mlp` | Pre-FFN LayerNorm | |
| `gate_mlp` | FFN output | `x + gate * ffn(x)` |

### Why "Zero"?
The gate parameters are **initialized to zero**, so at the start of training
the transformer blocks have no effect (identity function). This helps training
stability — the model gradually learns to use the blocks.

In [ ]:
class AdaLayerNormZero(nn.Module):
    """
    Adaptive Layer Normalization with zero-initialized gates.
    Produces 6 modulation params from the conditioning vector.
    """

    def __init__(self, embedding_dim: int):
        super().__init__()
        self.silu = nn.SiLU()
        self.linear = nn.Linear(embedding_dim, 6 * embedding_dim, bias=True)
        self.norm = nn.LayerNorm(embedding_dim, elementwise_affine=False, eps=1e-6)

    def forward(
        self, x: torch.Tensor, emb: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        x:   (B, N, dim) input tokens
        emb: (B, dim)    conditioning vector

        Returns: (modulated_x, gate_msa, shift_mlp, scale_mlp, gate_mlp)
        """
        # Project conditioning to 6 * dim modulation parameters
        params = self.linear(self.silu(emb))
        shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = params.chunk(6, dim=-1)

        # Apply adaptive normalization: LN(x) * (1 + scale) + shift
        x = self.norm(x) * (1 + scale_msa[:, None, :]) + shift_msa[:, None, :]
        return x, gate_msa, shift_mlp, scale_mlp, gate_mlp


class AdaLayerNormContinuous(nn.Module):
    """
    Simpler adaptive layer norm: only scale + shift (no gates).
    Used for the final output norm and the last block's text norm.
    """

    def __init__(self, embedding_dim: int, conditioning_dim: int):
        super().__init__()
        self.silu = nn.SiLU()
        self.linear = nn.Linear(conditioning_dim, embedding_dim * 2, bias=True)
        self.norm = nn.LayerNorm(embedding_dim, elementwise_affine=False, eps=1e-6)

    def forward(self, x: torch.Tensor, conditioning: torch.Tensor) -> torch.Tensor:
        emb = self.linear(self.silu(conditioning))
        scale, shift = emb.chunk(2, dim=-1)
        x = self.norm(x) * (1 + scale[:, None, :]) + shift[:, None, :]
        return x

## 9. FeedForward Network

SD3 uses **GELU(tanh)** activation — *not* GeGLU like SD 1.x/2.x.
This means the FFN is a simple two-layer MLP with 4× expansion:

```
x (1536) -> Linear (1536 -> 6144) -> GELU(tanh) -> Linear (6144 -> 1536)
```

No gating mechanism — the gating is handled by AdaLN-Zero instead.

In [ ]:
class ApproximateGELU(nn.Module):
    """
    Linear + GELU(tanh) combined into one module.
    Diffusers weight keys: proj.weight, proj.bias
    """

    def __init__(self, dim_in: int, dim_out: int):
        super().__init__()
        self.proj = nn.Linear(dim_in, dim_out)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.gelu(self.proj(x), approximate="tanh")


class FeedForward(nn.Module):
    """
    Two-layer MLP with GELU(tanh) activation.

    Uses nn.ModuleList to match diffusers naming:
      net.0 = ApproximateGELU (Linear + GELU)
      net.1 = Dropout (no-op at inference)
      net.2 = Linear (output projection)
    """

    def __init__(self, dim: int, mult: int = 4):
        super().__init__()
        inner_dim = dim * mult
        self.net = nn.ModuleList([
            ApproximateGELU(dim, inner_dim),    # net.0
            nn.Dropout(0.0),                    # net.1
            nn.Linear(inner_dim, dim),          # net.2
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.net:
            x = layer(x)
        return x

## 10. Joint Attention — The Core of MMDiT

This is the **key innovation** in SD3. Instead of separate self-attention
(image-to-image) and cross-attention (image-queries-text) like SD 1.x/2.x,
MMDiT uses **joint self-attention** over the concatenated sequence:

```
Image tokens: (B, N_img, 1536)    Text tokens: (B, N_txt, 1536)
     |                                  |
     |- Q, K, V projections             |- Q, K, V projections (separate weights)
     |- QK RMSNorm (per-head)           |- QK RMSNorm (per-head)
     |                                  |
     |------Concatenate -------------|
                    |
            Joint Self-Attention
         (every token sees every token)
                    |
     |-------Split ---------|
     |                         |
 Image output proj          Text output proj
```

### Why joint attention?
- **Symmetric**: Both modalities influence each other equally (text can
  attend to image AND image can attend to text).
- **Simpler**: One attention operation instead of two (self + cross).
- **Better quality**: Empirically produces better prompt adherence and
  text rendering in generated images.

### QK Normalization
Both image and text Q/K vectors are **RMS-normalized per head** before
computing attention. This prevents attention logit explosion during
training (a common failure mode for large transformers).

### Last Block Special Case
The final transformer block sets `context_pre_only=True`: text tokens
still contribute to the joint attention computation, but no text output
is produced (since we only need the image tokens for the final prediction).

In [ ]:
class JointAttention(nn.Module):
    """
    MMDiT Joint Attention: image and text tokens attend to each other
    in a single self-attention operation.

    Each modality has its own Q/K/V projections and QK RMSNorms,
    but the actual attention computation is shared (joint).
    """

    def __init__(
        self,
        dim: int,
        num_heads: int,
        head_dim: int,
        context_dim: Optional[int] = None,
        context_pre_only: bool = False,  # True for the last block
    ):
        super().__init__()
        inner_dim = num_heads * head_dim
        context_dim = context_dim or dim

        self.heads = num_heads
        self.head_dim = head_dim
        self.context_pre_only = context_pre_only

        # Image token projections
        self.to_q = nn.Linear(dim, inner_dim, bias=True)
        self.to_k = nn.Linear(dim, inner_dim, bias=True)
        self.to_v = nn.Linear(dim, inner_dim, bias=True)

        # Text token projections (separate weights!)
        self.add_q_proj = nn.Linear(context_dim, inner_dim, bias=True)
        self.add_k_proj = nn.Linear(context_dim, inner_dim, bias=True)
        self.add_v_proj = nn.Linear(context_dim, inner_dim, bias=True)

        # QK normalization (per-head RMSNorm, dim=64)
        self.norm_q = RMSNorm(head_dim)
        self.norm_k = RMSNorm(head_dim)
        self.norm_added_q = RMSNorm(head_dim)  # Text Q norm
        self.norm_added_k = RMSNorm(head_dim)  # Text K norm

        # Output projections
        self.to_out = nn.ModuleList([
            nn.Linear(inner_dim, dim, bias=True),
            nn.Dropout(0.0),
        ])

        # Text output (not present in the last block)
        if not context_pre_only:
            self.to_add_out = nn.Linear(inner_dim, dim, bias=True)

    def forward(
        self,
        hidden_states: torch.Tensor,          # (B, N_img, dim)
        encoder_hidden_states: torch.Tensor,  # (B, N_txt, dim)
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        B = hidden_states.shape[0]
        N_txt = encoder_hidden_states.shape[1]
        h, d = self.heads, self.head_dim

        # Project image and text tokens separately
        q     = self.to_q(hidden_states).view(B, -1, h, d)
        k     = self.to_k(hidden_states).view(B, -1, h, d)
        v     = self.to_v(hidden_states).view(B, -1, h, d)
        q_ctx = self.add_q_proj(encoder_hidden_states).view(B, -1, h, d)
        k_ctx = self.add_k_proj(encoder_hidden_states).view(B, -1, h, d)
        v_ctx = self.add_v_proj(encoder_hidden_states).view(B, -1, h, d)

        # QK normalization (per-head) for training stability
        q     = self.norm_q(q)
        k     = self.norm_k(k)
        q_ctx = self.norm_added_q(q_ctx)
        k_ctx = self.norm_added_k(k_ctx)

        # Concatenate text + image for joint attention
        # Text first, then image (order matters for splitting later)
        q_joint = torch.cat([q_ctx, q], dim=1).transpose(1, 2)  # (B, h, N_total, d)
        k_joint = torch.cat([k_ctx, k], dim=1).transpose(1, 2)
        v_joint = torch.cat([v_ctx, v], dim=1).transpose(1, 2)

        # Joint self-attention (Flash Attention when available)
        out = F.scaled_dot_product_attention(
            q_joint, k_joint, v_joint,
            dropout_p=0.0, is_causal=False,
        )

        # Reshape: (B, h, N_total, d) -> (B, N_total, h*d)
        out = out.transpose(1, 2).contiguous().view(B, -1, h * d)

        # Split back into text and image
        ctx_out = out[:, :N_txt, :]
        img_out = out[:, N_txt:, :]

        # Separate output projections for each modality
        img_out = self.to_out[0](img_out)
        img_out = self.to_out[1](img_out)  # Dropout (no-op)

        if not self.context_pre_only:
            ctx_out = self.to_add_out(ctx_out)
        else:
            ctx_out = None  # Last block: no text output needed

        return img_out, ctx_out

## 11. Joint Transformer Block (One MMDiT Layer)

Each MMDiT block processes **two parallel streams** (image and text) that
interact through joint attention:

```
Image Stream:                          Text Stream:
-------------------                 --------------------
AdaLN-Zero(img, cond)                  AdaLN-Zero(txt, cond)
     |                                       |
     |----Joint Attention ----------------|
     |                                       |
 gate_msa * attn_out                   gate_msa * attn_out
     + residual                              + residual
     |                                       |
 AdaLN-mod + FFN                       AdaLN-mod + FFN
     |                                       |
 gate_mlp * ffn_out                    gate_mlp * ffn_out
     + residual                             + residual
```

The two streams share attention but have **separate** normalizations, gates,
and FFNs. This lets each modality process information differently while still
communicating through the shared attention.

In [ ]:
class JointTransformerBlock(nn.Module):
    """
    One MMDiT block: dual-stream (image + text) with shared joint attention.

    The last block (context_pre_only=True) skips the text FFN and output,
    since only image tokens are needed for the final prediction.
    """

    def __init__(
        self,
        dim: int,
        num_attention_heads: int,
        attention_head_dim: int,
        context_pre_only: bool = False,
    ):
        super().__init__()
        self.context_pre_only = context_pre_only

        # Image stream norms
        self.norm1 = AdaLayerNormZero(dim)  # Pre-attention
        self.norm2 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)  # Pre-FFN

        # Text stream norms
        if context_pre_only:
            # Last block: simpler norm (no gates needed for text)
            self.norm1_context = AdaLayerNormContinuous(dim, dim)
        else:
            self.norm1_context = AdaLayerNormZero(dim)
            self.norm2_context = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)

        # Shared joint attention
        self.attn = JointAttention(
            dim=dim,
            num_heads=num_attention_heads,
            head_dim=attention_head_dim,
            context_dim=dim,
            context_pre_only=context_pre_only,
        )

        # Separate FFNs for each stream
        self.ff = FeedForward(dim)  # Image FFN
        if not context_pre_only:
            self.ff_context = FeedForward(dim)  # Text FFN

    def forward(
        self,
        hidden_states: torch.Tensor,         # (B, N_img, dim)
        encoder_hidden_states: torch.Tensor,  # (B, N_txt, dim)
        temb: torch.Tensor,                   # (B, dim) conditioning
    ) -> Tuple[torch.Tensor, torch.Tensor]:

        # AdaLN normalization for both streams
        norm_img, gate_msa, shift_mlp, scale_mlp, gate_mlp = self.norm1(
            hidden_states, temb
        )

        if self.context_pre_only:
            norm_ctx = self.norm1_context(encoder_hidden_states, temb)
        else:
            norm_ctx, c_gate_msa, c_shift_mlp, c_scale_mlp, c_gate_mlp = (
                self.norm1_context(encoder_hidden_states, temb)
            )

        # Joint Attention (both streams interact)
        attn_img, attn_ctx = self.attn(
            hidden_states=norm_img,
            encoder_hidden_states=norm_ctx,
        )

        # Image stream: gated attention residual + modulated FFN
        hidden_states = hidden_states + gate_msa[:, None, :] * attn_img

        norm_img = self.norm2(hidden_states)
        norm_img = norm_img * (1 + scale_mlp[:, None, :]) + shift_mlp[:, None, :]
        hidden_states = hidden_states + gate_mlp[:, None, :] * self.ff(norm_img)

        # Text stream (skip in last block)
        if attn_ctx is not None:
            encoder_hidden_states = (
                encoder_hidden_states + c_gate_msa[:, None, :] * attn_ctx
            )
            norm_ctx = self.norm2_context(encoder_hidden_states)
            norm_ctx = norm_ctx * (1 + c_scale_mlp[:, None, :]) + c_shift_mlp[:, None, :]
            encoder_hidden_states = (
                encoder_hidden_states + c_gate_mlp[:, None, :] * self.ff_context(norm_ctx)
            )

        return encoder_hidden_states, hidden_states

## 12. Full SD3 Transformer

Putting it all together:

```
Inputs:
  latent:          (B, 16, H, W)     - from SD3 VAE
  text_tokens:     (B, seq, 4096)    - from T5-XXL
  pooled_text:     (B, 2048)         - from CLIP-L + OpenCLIP-bigG
  timestep:        (B,)              - float in [0, 1]

Pipeline:
  1. PatchEmbed(latent)           => (B, N_patches, 1536)  image tokens
  2. CombinedEmbed(t, pooled)     => (B, 1536)             conditioning
  3. Linear(text_tokens)          => (B, seq, 1536)        text tokens
  4. 24× JointTransformerBlock    => process both streams
  5. AdaLN + Linear projection    => (B, N_patches, 64)    per-patch output
  6. Unpatchify                   => (B, 16, H, W)         predicted velocity

Output:
  predicted velocity (for rectified flow), same shape as input latent
```

### Rectified Flow vs DDPM
SD 1.x/2.x used DDPM where the model predicts **noise ε** (what was added).
SD3 uses rectified flow where the model predicts a **velocity v** (the
direction to move from noise toward the clean image). The sampling process
is simpler: essentially linear interpolation guided by the velocity field.

In [ ]:
class SD3Transformer2DModel(nn.Module):
    """
    Stable Diffusion 3 MMDiT Transformer.

    Replaces the U-Net from SD 1.x/2.x with a pure transformer.
    Predicts a velocity field (for rectified flow) rather than noise.
    """

    def __init__(self, config: SD3TransformerConfig):
        super().__init__()
        self.config = config
        self.out_channels = config.out_channels
        inner_dim = config.num_attention_heads * config.attention_head_dim  # 24 * 64 = 1536

        # Patch embedding: latent -> token sequence
        self.pos_embed = PatchEmbed(
            sample_size=config.sample_size,
            patch_size=config.patch_size,
            in_channels=config.in_channels,
            embed_dim=inner_dim,
            pos_embed_max_size=config.pos_embed_max_size,
        )

        # Conditioning: timestep + pooled text
        self.time_text_embed = CombinedTimestepTextProjEmbeddings(
            embedding_dim=inner_dim,
            pooled_projection_dim=config.pooled_projection_dim,
        )

        # Text context projection: T5-XXL (4096) -> inner_dim (1536)
        self.context_embedder = nn.Linear(
            config.joint_attention_dim,
            config.caption_projection_dim,
            bias=True,
        )

        # 24 MMDiT blocks (last one is context_pre_only)
        self.transformer_blocks = nn.ModuleList([
            JointTransformerBlock(
                dim=inner_dim,
                num_attention_heads=config.num_attention_heads,
                attention_head_dim=config.attention_head_dim,
                context_pre_only=(i == config.num_layers - 1),
            )
            for i in range(config.num_layers)
        ])

        # Final output: AdaLN + linear to patch pixels
        self.norm_out = AdaLayerNormContinuous(inner_dim, inner_dim)
        self.proj_out = nn.Linear(
            inner_dim,
            config.patch_size * config.patch_size * config.out_channels,  # 2*2*16 = 64
            bias=True,
        )

    def forward(
        self,
        hidden_states: torch.Tensor,         # (B, 16, H, W) latent
        encoder_hidden_states: torch.Tensor,  # (B, seq, 4096) T5-XXL tokens
        pooled_projections: torch.Tensor,     # (B, 2048) pooled CLIP+OpenCLIP
        timestep: torch.Tensor,               # (B,) float timestep
    ) -> torch.Tensor:
        height, width = hidden_states.shape[2], hidden_states.shape[3]

        # 1. Patchify + positional embedding
        hidden_states = self.pos_embed(hidden_states)  # (B, N_patches, 1536)

        # 2. Compute conditioning vector
        temb = self.time_text_embed(timestep, pooled_projections)  # (B, 1536)

        # 3. Project text context from T5 dim to inner dim
        encoder_hidden_states = self.context_embedder(encoder_hidden_states)  # (B, seq, 1536)

        # 4. Process through all transformer blocks
        for block in self.transformer_blocks:
            encoder_hidden_states, hidden_states = block(
                hidden_states=hidden_states,
                encoder_hidden_states=encoder_hidden_states,
                temb=temb,
            )

        # 5. Final norm + projection to patch pixels
        hidden_states = self.norm_out(hidden_states, temb)
        hidden_states = self.proj_out(hidden_states)  # (B, N, patch^2 * out_ch)

        # 6. Unpatchify: tokens -> spatial
        p = self.config.patch_size
        h = height // p
        w = width // p
        hidden_states = hidden_states.reshape(
            hidden_states.shape[0], h, w, p, p, self.out_channels
        )
        # Rearrange: (B, h, w, p1, p2, C) -> (B, C, h*p1, w*p2)
        hidden_states = hidden_states.permute(0, 5, 1, 3, 2, 4).contiguous()
        hidden_states = hidden_states.reshape(
            hidden_states.shape[0], self.out_channels, h * p, w * p,
        )

        return hidden_states  # (B, 16, H, W) predicted velocity

## 13. Weight Loading from HuggingFace

Like SD 2.x, our class names and structure match diffusers exactly, so
weights load directly with `load_state_dict()`. SD3 weights may be
**sharded** (split across multiple safetensors files) due to model size,
so the loader handles that case too.

**Note**: SD3 models require accepting a license on HuggingFace before download.

In [ ]:
def load_sd3_config_from_hf(
    repo_id: str, subfolder: str = "transformer",
) -> SD3TransformerConfig:
    """Load transformer config from a diffusers-format HF repo."""
    config_path = hf_hub_download(
        repo_id=repo_id, filename=f"{subfolder}/config.json",
    )
    with open(config_path, "r") as f:
        cfg = json.load(f)

    return SD3TransformerConfig(
        sample_size=cfg.get("sample_size", 128),
        patch_size=cfg.get("patch_size", 2),
        in_channels=cfg.get("in_channels", 16),
        out_channels=cfg.get("out_channels", 16),
        num_layers=cfg.get("num_layers", 24),
        num_attention_heads=cfg.get("num_attention_heads", 24),
        attention_head_dim=cfg.get("attention_head_dim", 64),
        joint_attention_dim=cfg.get("joint_attention_dim", 4096),
        caption_projection_dim=cfg.get("caption_projection_dim", 1536),
        pooled_projection_dim=cfg.get("pooled_projection_dim", 2048),
        pos_embed_max_size=cfg.get("pos_embed_max_size", 192),
    )


def load_sd3_weights_from_hf(
    model: nn.Module, repo_id: str, subfolder: str = "transformer",
) -> nn.Module:
    """
    Load pretrained weights. Tries:
    1. Single safetensors file
    2. Sharded safetensors (multiple files via index.json)
    3. PyTorch .bin fallback
    """
    state_dict = None

    # Try single safetensors
    try:
        weights_path = hf_hub_download(
            repo_id=repo_id,
            filename=f"{subfolder}/diffusion_pytorch_model.safetensors",
        )
        state_dict = load_file(weights_path)
    except Exception:
        print(f"Failed to load safetensor")

    # Try sharded safetensors
    if state_dict is None:
        try:
            index_path = hf_hub_download(
                repo_id=repo_id,
                filename=f"{subfolder}/diffusion_pytorch_model.safetensors.index.json",
            )
            with open(index_path) as f:
                index = json.load(f)

            shard_files = set(index["weight_map"].values())
            state_dict = {}
            for shard_file in shard_files:
                shard_path = hf_hub_download(
                    repo_id=repo_id, filename=f"{subfolder}/{shard_file}",
                )
                state_dict.update(load_file(shard_path))
        except Exception:
            print(f"Failed to update state_dit")

    # Try pytorch bin
    if state_dict is None:
        weights_path = hf_hub_download(
            repo_id=repo_id,
            filename=f"{subfolder}/diffusion_pytorch_model.bin",
        )
        state_dict = torch.load(weights_path, map_location="cpu")

    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print(f"Loaded weights from {repo_id}/{subfolder}")
    if not missing and not unexpected:
        print("\tAll weights loaded successfully!")
    else:
        if missing:
            print(f"\tMissing keys ({len(missing)}): {missing[:5]}...")
        if unexpected:
            print(f"\tUnexpected keys ({len(unexpected)}): {unexpected[:5]}...")

    return model

## 14. Build and Test

Let's build the SD3-Medium transformer with default config and run a
forward pass with dummy inputs to verify everything works.

**This only builds the MMDiT transformer.** A full SD3 pipeline would also need:
- SD3 VAE (16-channel encoder/decoder)
- CLIP-L text encoder
- OpenCLIP-bigG text encoder
- T5-XXL text encoder
- Rectified flow sampler

In [ ]:
# Build model with default SD3-Medium config
config = SD3TransformerConfig()
model = SD3Transformer2DModel(config)

total_params = sum(p.numel() for p in model.parameters())
print(f"SD3-Medium MMDiT Transformer")
print(f"\tLayers: {config.num_layers}")
print(f"\tInner dim: {config.num_attention_heads * config.attention_head_dim}")
print(f"\tHeads: {config.num_attention_heads} x {config.attention_head_dim}")
print(f"\tParameters: {total_params:,} ({total_params / 1e6:.0f}M)")

In [ ]:
# Forward pass with dummy inputs
device = "cpu"
dtype = torch.float32

model.to(device=device, dtype=dtype)
model.eval()

batch = 1

# SD3 latent: 16 channels, spatial = image_size / 8
# For 512x512 image: 64x64 latent; for 1024x1024: 128x128
latent_h, latent_w = 64, 64
latent = torch.randn(batch, 16, latent_h, latent_w, device=device, dtype=dtype)

# Text: T5-XXL hidden states (sequence of 4096-dim tokens)
text_seq_len = 154  # CLIP (77) + T5 (77) tokens typically
encoder_hidden_states = torch.randn(
    batch, text_seq_len, 4096, device=device, dtype=dtype
)

# Pooled text: CLIP-L (768) + OpenCLIP-bigG (1280) = 2048
pooled_projections = torch.randn(batch, 2048, device=device, dtype=dtype)

# Timestep: float for rectified flow (0.0 = clean, 1.0 = noise)
timestep = torch.tensor([0.5], device=device, dtype=dtype)

print("Input shapes:")
print(f"\tlatent:          {latent.shape}")
print(f"\ttext embeddings: {encoder_hidden_states.shape}")
print(f"\tpooled text:     {pooled_projections.shape}")
print(f"\ttimestep:        {timestep.shape} (value={timestep.item()})")

with torch.inference_mode():
    output = model(latent, encoder_hidden_states, pooled_projections, timestep)

print(f"\nOutput shape: {output.shape}")
print(f"Expected:     ({batch}, 16, {latent_h}, {latent_w})")
assert output.shape == (batch, 16, latent_h, latent_w)
print("Forward pass successful!")

In [ ]:
# Optional: Load pretrained weights from HuggingFace
# SD3 models require accepting a license on HuggingFace.
# After accepting, set: export HUGGING_FACE_HUB_TOKEN=hf_xxxxx
#
# Uncomment below to load real weights:

# repo_id = "stabilityai/stable-diffusion-3-medium-diffusers"
# config = load_sd3_config_from_hf(repo_id, subfolder="transformer")
# model = SD3Transformer2DModel(config)
# model = load_sd3_weights_from_hf(model, repo_id, subfolder="transformer")
# model.eval()
#
# # GPU inference:
# model.to(device="cuda", dtype=torch.float16)
# latent = latent.to("cuda", torch.float16)
# encoder_hidden_states = encoder_hidden_states.to("cuda", torch.float16)
# pooled_projections = pooled_projections.to("cuda", torch.float16)
#
# with torch.inference_mode():
#     output = model(latent, encoder_hidden_states, pooled_projections, timestep.cuda())
# print("Output:", output.shape)

## Summary: Evolution from SD 1.5 to SD 3.x

| Component | SD 1.5 | SD 2.x | SD 3.x |
|---|---|---|---|
| **Backbone** | UNet (conv + attn) | UNet (conv + attn) | **Transformer** (DiT) |
| **Image repr** | Conv feature maps | Conv feature maps | **Patch tokens** (ViT) |
| **Text-image interaction** | Cross-attention | Cross-attention | **Joint attention** |
| **Conditioning** | Timestep add | Timestep add | **AdaLN-Zero** (shift/scale/gate) |
| **Normalization** | GroupNorm | GroupNorm | LayerNorm + **RMSNorm** (QK) |
| **FFN activation** | GeGLU | GeGLU | **GELU(tanh)** |
| **Text encoder** | CLIP ViT-L (768) | OpenCLIP ViT-H (1024) | **CLIP-L + OpenCLIP-bigG + T5-XXL** |
| **Latent channels** | 4 | 4 | **16** |
| **Noise schedule** | DDPM (ε prediction) | DDPM (ε prediction) | **Rectified Flow** (v prediction) |
| **Skip connections** | U-Net skip (essential) | U-Net skip (essential) | **None** (pure transformer) |
| **Weight format** | Single .ckpt + key mapping | Diffusers safetensors | Diffusers safetensors (sharded) |

The shift from UNet to Transformer is the most significant architectural change
in the Stable Diffusion family. It enables better scaling (transformers scale
more predictably than UNets), simpler architecture (no skip connections to tune),
and more natural multi-modal interaction (joint attention vs asymmetric cross-attention).

## 15. Full Generation Pipeline

Now we wire the MMDiT transformer together with SD3's VAE and three text
encoders into a complete txt2img pipeline.

```
  Prompt
      |
      |-► CLIP-L (768)   --┐
      |-► CLIP-G (1280)  --┤ concat pools => (2048,)   pooled_projections
      └-► T5-XXL (4096)  --┘

  CLIP-L tokens (77, 768=>pad=>4096) -┐
  CLIP-G tokens (77, 1280=>pad=>4096) -┤ concat => (154+T5_len, 4096)
  T5 tokens     (T5_len, 4096)     --┘          encoder_hidden_states

  Pure noise z_T  (B, 16, H/8, W/8)         ← 1024px => 128×128 latent
      |
      ▼  Rectified Flow sampler (20 steps)
  SD3Transformer2DModel --► velocity v̂
      |
      ▼
  Clean latent z_0  (B, 16, H/8, W/8)
      |
      ▼
  VAE Decoder  --► RGB image  (B, 3, H, W)
```

### Key Differences from SD 1.5 / 2.x

| Aspect | SD 1.5/2.x | SD 3.x |
|---|---|---|
| Denoising backbone | U-Net (conv) | **MMDiT (transformer)** |
| Noise schedule | DDPM/DDIM (ε-prediction) | **Rectified Flow (v-prediction)** |
| Latent channels | 4 | **16** |
| VAE scale factor | 0.18215 | **1.5305** |
| Text encoders | 1 (CLIP) | **3 (CLIP-L + CLIP-G + T5-XXL)** |
| Text token dim | 768/1024 | **4096 (padded)** |
| Pooled text dim | — | **2048 (CLIP-L pool + CLIP-G pool)** |

> **Note**: SD3 models require accepting the license on HuggingFace.
> Run `huggingface-cli login` and set `HUGGING_FACE_HUB_TOKEN` before downloading.

In [ ]:
# !pip install diffusers transformers accelerate sentencepiece

### 15a. Rectified Flow Sampler

SD3 uses **Rectified Flow** (Liu et al. 2022) instead of DDPM.

The key difference:
- DDPM/DDIM: predicts **noise** ε added to the image
- Rectified Flow: predicts **velocity** v = x_1 − x_0 (direction from noise x_1 to clean x_0)

```
  Forward process:  x_t = (1 − t) · x_0 + t · x_1
                         = (1 − t) · clean + t · noise,   t ∈ [0, 1]

  Reverse step:     x_{t−Δt} = x_t − v̂ · Δt
```

Compared to DDPM:
- The timestep scale is [0, 1] (not [0, 999])
- Trajectories are **straight lines** in expectation => fewer steps needed
- Allows high-quality generation in 20–28 steps

In [ ]:
class RectifiedFlowSampler:
    """
    Euler sampler for Rectified Flow (SD3 / SD3.5).

    The model predicts the velocity v = x_1 - x_0 (noise direction minus
    clean image direction). We integrate this with simple Euler steps:

        x_{t - dt} = x_t - v * dt

    Timestep convention: t=1.0 is pure noise, t=0.0 is clean image.

    Reference: Flow Matching for Generative Modeling (Lipman et al. 2022)
               Scaling Rectified Flow Transformers (Esser et al. 2024 / SD3)
    """

    def __init__(self, num_train_timesteps: int = 1000):
        # Evenly-spaced sigma schedule: sigma=t/T, mapping [0,T-1] => [0,1]
        self.num_train_timesteps = num_train_timesteps
        self.timesteps = None
        self.sigmas = None

    def set_timesteps(self, num_inference_steps: int, device="cpu"):
        """
        Build evenly-spaced timesteps from 1.0 (noise) to 0.0 (clean).
        SD3 uses a shift of 3.0 for better quality at high resolutions.
        """
        # Uniform timesteps on [0, 1], descending (noise => clean)
        ts = torch.linspace(1.0, 0.0, num_inference_steps + 1, device=device)
        self.sigmas = ts                          # t values at boundaries
        self.timesteps = ts[:-1]                  # leading edges of intervals
        self.dt = ts[:-1] - ts[1:]               # step sizes (all positive)
        self.num_inference_steps = num_inference_steps

    def step(
        self,
        step_index: int,
        latents: torch.Tensor,
        velocity_pred: torch.Tensor,
    ) -> torch.Tensor:
        """
        One Euler step of the reverse ODE:
            x_{t - dt} = x_t - v * dt

        Parameters
       -------
        step_index : int
            Current step (0 = noisiest, num_inference_steps-1 = cleanest).
        latents : Tensor (B, 16, H, W)
            Current noisy latent x_t.
        velocity_pred : Tensor (B, 16, H, W)
            Model's predicted velocity v̂.

        Returns
       ----
        Tensor (B, 16, H, W) — less noisy latent x_{t-dt}
        """
        dt = self.dt[step_index].to(latents.device)
        return latents - velocity_pred * dt

    def scale_noise(
        self, latents: torch.Tensor, t: float, generator=None
    ) -> torch.Tensor:
        """
        Forward process: x_t = (1-t)·x_0 + t·ε
        Used for img2img to noise an encoded image up to timestep t.
        """
        noise = torch.randn(
            latents.shape, generator=generator,
            device=latents.device, dtype=latents.dtype,
        )
        return (1 - t) * latents + t * noise

### 15b. Text Encoding for SD3

SD3 uses **three** text encoders whose outputs are concatenated:

```
  CLIP-L (ViT-L/14):
      tokens (77,) => hidden states (77, 768)  => zero-pad => (77, 4096)
                   => pooler_output (768,)     -------------------------┐
                                                                        ▼
  CLIP-G (ViT-bigG/14):                                          cat => (2048,)
      tokens (77,) => hidden states (77, 1280) => zero-pad => (77, 4096)  |
                   => text_embeds  (1280,)     -------------------------┘
                                                   pooled_projections

  T5-XXL (optional — set use_t5=False to save ~10 GB VRAM):
      tokens (256,) => hidden states (256, 4096) -┐
                                                  ▼
  Final:  cat([clip_l_pad, clip_g_pad, t5], dim=1)  => (77+77+256, 4096)
                                               encoder_hidden_states
```

In [ ]:
from transformers import (
    CLIPTextModelWithProjection,
    CLIPTokenizer,
    T5EncoderModel,
    T5TokenizerFast,
)
import torch.nn.functional as F


def encode_prompt_sd3(
    prompt: str,
    neg_prompt: str,
    clip_l,
    clip_l_tok,
    clip_g,
    clip_g_tok,
    t5=None,
    t5_tok=None,
    use_t5: bool = True,
    device: str = "cuda",
):
    """
    Encode prompts for SD3 using up to three text encoders.

    Returns
   ----
    (encoder_hidden_states, pooled_projections) — each is a tuple
    (positive, negative) stacked as (2, seq, 4096) and (2, 2048).
    """
    max_seq = 77  # CLIP context window
    t5_max_seq = 256 if use_t5 and t5 is not None else 0
    target_dim = 4096  # All CLIP embeddings are zero-padded to this

    def _tok(tokenizer, text, max_length):
        return tokenizer(
            text, padding="max_length", max_length=max_length,
            truncation=True, return_tensors="pt",
        ).input_ids.to(device)

    results = {}
    for label, text in [("pos", prompt), ("neg", neg_prompt)]:
        # CLIP-L --------------------------------------------------
        ids_l = _tok(clip_l_tok, text, max_seq)
        out_l = clip_l(ids_l, output_hidden_states=True)
        # Use second-to-last hidden state (matches diffusers convention)
        seq_l = out_l.hidden_states[-2]                       # (1, 77, 768)
        pool_l = out_l.text_embeds if hasattr(out_l, "text_embeds") else out_l.pooler_output  # (1, 768)

        # CLIP-G --------------------------------------------------
        ids_g = _tok(clip_g_tok, text, max_seq)
        out_g = clip_g(ids_g, output_hidden_states=True)
        seq_g = out_g.hidden_states[-2]                       # (1, 77, 1280)
        pool_g = out_g.text_embeds if hasattr(out_g, "text_embeds") else out_g.pooler_output  # (1, 1280)

        # Pad CLIP sequences to target_dim=4096 -------------------
        seq_l_pad = F.pad(seq_l, (0, target_dim - seq_l.shape[-1]))   # (1, 77, 4096)
        seq_g_pad = F.pad(seq_g, (0, target_dim - seq_g.shape[-1]))   # (1, 77, 4096)

        # Pooled (concat CLIP-L + CLIP-G pools) -------------------
        pooled = torch.cat([pool_l, pool_g], dim=-1)  # (1, 2048)

        # T5 (optional) --------------------------------------------
        clip_seq = torch.cat([seq_l_pad, seq_g_pad], dim=1)  # (1, 154, 4096)

        if use_t5 and t5 is not None and t5_tok is not None:
            ids_t5 = _tok(t5_tok, text, t5_max_seq)
            with torch.no_grad():
                seq_t5 = t5(ids_t5).last_hidden_state  # (1, 256, 4096)
            encoder_seq = torch.cat([clip_seq, seq_t5], dim=1)  # (1, 410, 4096)
        else:
            encoder_seq = clip_seq  # (1, 154, 4096) — CLIP only

        results[label] = (encoder_seq, pooled)

    # Stack positive + negative for CFG
    enc_hidden = torch.cat([results["pos"][0], results["neg"][0]], dim=0)  # (2, seq, 4096)
    pooled = torch.cat([results["pos"][1], results["neg"][1]], dim=0)       # (2, 2048)
    return enc_hidden, pooled

### 15c. Load All Pipeline Components

We load from `stabilityai/stable-diffusion-3-medium-diffusers`.
**Requires accepting the license on HuggingFace.**

> To skip T5-XXL (saves ~10 GB VRAM, slight quality reduction):
> set `use_t5 = False` in the config below.

In [ ]:
from diffusers import AutoencoderKL
from huggingface_hub import hf_hub_download

SD3_REPO = "stabilityai/stable-diffusion-3-medium-diffusers"

# Set to False if you have < 24 GB VRAM — T5-XXL adds ~10 GB
USE_T5 = True

print("Loading CLIP-L text encoder...")
clip_l = CLIPTextModelWithProjection.from_pretrained(
    SD3_REPO, subfolder="text_encoder", torch_dtype=torch.float16
).eval()
clip_l_tok = CLIPTokenizer.from_pretrained(SD3_REPO, subfolder="tokenizer")

print("Loading CLIP-G (OpenCLIP ViT-bigG) text encoder...")
clip_g = CLIPTextModelWithProjection.from_pretrained(
    SD3_REPO, subfolder="text_encoder_2", torch_dtype=torch.float16
).eval()
clip_g_tok = CLIPTokenizer.from_pretrained(SD3_REPO, subfolder="tokenizer_2")

t5, t5_tok = None, None
if USE_T5:
    print("Loading T5-XXL encoder (this is large — ~10 GB)...")
    t5 = T5EncoderModel.from_pretrained(
        SD3_REPO, subfolder="text_encoder_3", torch_dtype=torch.float16
    ).eval()
    t5_tok = T5TokenizerFast.from_pretrained(SD3_REPO, subfolder="tokenizer_3")
else:
    print("Skipping T5-XXL (USE_T5=False)")

# SD3 VAE
# SD3 VAE: 16-channel latents (vs 4 in SD1.5/2.x), 8× compression
print("Loading VAE (16-channel)...")
vae_sd3 = AutoencoderKL.from_pretrained(
    SD3_REPO, subfolder="vae", torch_dtype=torch.float16
).eval()

# Our MMDiT transformer
# Built and loaded in Section 13-14 above; just confirm it's ready
print(f"Transformer ready: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M params")

# SD3 constants
VAE_SCALE_SD3 = 1.5305   # Trained with this shift (different from SD1.5's 0.18215)
VAE_SHIFT_SD3 = 0.0609   # Additional shift for SD3 VAE
SD3_WIDTH  = 1024
SD3_HEIGHT = 1024
SD3_LATENT_H = SD3_HEIGHT // 8   # 128
SD3_LATENT_W = SD3_WIDTH  // 8   # 128

print(f"\nAll SD3 components loaded.")
print(f"\tResolution : {SD3_WIDTH}×{SD3_HEIGHT}")
print(f"\tLatent     : {SD3_LATENT_H}×{SD3_LATENT_W}×16 channels")
print(f"\tT5-XXL     : {'enabled' if USE_T5 else 'disabled (CLIP-only mode)'}")

### 15d. Generate Function

The SD3 generate function follows the same structure as SD 1.5/2.x but uses:
- Three text encoders => concatenated 4096-dim sequence tokens
- 16-channel VAE (latents have 16 channels, not 4)
- Rectified Flow sampler (velocity prediction, not noise prediction)

In [ ]:
from tqdm import tqdm
from PIL import Image

def generate_sd3(
    prompt: str,
    negative_prompt: str = "",
    cfg_scale: float = 7.0,
    num_inference_steps: int = 28,
    seed: int = 42,
    width: int = SD3_WIDTH,
    height: int = SD3_HEIGHT,
    use_t5: bool = USE_T5,
    device: str = "cuda",
) -> Image.Image:
    """
    Full SD 3 txt2img generation pipeline.

    Components
   -------
    clip_l / clip_g / t5  : Three text encoders (loaded above)
    vae_sd3               : 16-channel AutoencoderKL (loaded above)
    model                 : SD3Transformer2DModel (built in Section 13-14)
    RectifiedFlowSampler  : Euler ODE integrator (defined above)

    Parameters
   -------
    prompt : str
        Positive text description.
    negative_prompt : str
        What to suppress (empty string = no negative guidance).
    cfg_scale : float
        Classifier-free guidance scale (default 7.0 for SD3).
    num_inference_steps : int
        Euler integration steps (28 is SD3's default).
    seed : int
        RNG seed.
    width / height : int
        Output resolution (must be divisible by 8; default 1024×1024).
    use_t5 : bool
        Whether to include T5-XXL tokens (set False for lower VRAM).
    device : str
        \"cuda\" or \"cpu\".

    Returns
   ----
    PIL.Image  (width × height)
    """
    latent_h = height // 8
    latent_w = width  // 8
    generator = torch.Generator(device=device).manual_seed(seed)

    # Step 1: Encode text --------------------------------------------
    for enc in [clip_l, clip_g] + ([t5] if t5 else []):
        enc.to(device)

    with torch.no_grad():
        enc_hidden, pooled = encode_prompt_sd3(
            prompt, negative_prompt,
            clip_l, clip_l_tok,
            clip_g, clip_g_tok,
            t5=t5, t5_tok=t5_tok,
            use_t5=use_t5 and t5 is not None,
            device=device,
        )
    # enc_hidden : (2, seq_len, 4096) — [positive, negative]
    # pooled     : (2, 2048)

    for enc in [clip_l, clip_g] + ([t5] if t5 else []):
        enc.to("cpu")

    # Step 2: Build sampler
    sampler = RectifiedFlowSampler()
    sampler.set_timesteps(num_inference_steps, device=device)

    # Step 3: Initial latent (pure noise)
    # SD3 has 16-channel latents
    latents = torch.randn(
        (1, 16, latent_h, latent_w),
        generator=generator, device=device, dtype=torch.float16,
    )

    # Step 4: Denoising loop (Rectified Flow Euler)
    transformer = model.half().to(device)
    enc_hidden = enc_hidden.half()
    pooled = pooled.half()

    with torch.no_grad():
        for i, t in enumerate(tqdm(sampler.timesteps, desc="SD3 denoising")):
            # Batch for CFG: [positive, negative]
            latent_in = latents.repeat(2, 1, 1, 1)  # (2, 16, H, W)
            t_batch = t.reshape(1).repeat(2).to(device)

            # MMDiT predicts velocity v
            velocity_pred = transformer(
                latent_in,
                encoder_hidden_states=enc_hidden,
                pooled_projections=pooled,
                timestep=t_batch,
            )  # (2, 16, H, W)

            # Classifier-Free Guidance
            vel_pos, vel_neg = velocity_pred.chunk(2)
            velocity_guided = vel_neg + cfg_scale * (vel_pos - vel_neg)

            # Euler step: x_{t-dt} = x_t − vv · dt
            latents = sampler.step(i, latents, velocity_guided)

    transformer.to("cpu")

    # Step 5: Decode latents => image
    # SD3 VAE uses a different scale+shift from SD 1.5/2.x
    vae_sd3.to(device)
    with torch.no_grad():
        # Reverse the scaling applied during encoding
        latents_dec = (latents / VAE_SCALE_SD3) + VAE_SHIFT_SD3
        images = vae_sd3.decode(latents_dec).sample        # (1, 3, H, W) in [-1, 1]
    vae_sd3.to("cpu")

    images = (images / 2 + 0.5).clamp(0, 1)
    images = (images * 255).round().to(torch.uint8)
    images = images.permute(0, 2, 3, 1).cpu().numpy()     # (1, H, W, 3)

    return Image.fromarray(images[0])

### 15e. Text-to-Image Inference

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

output = generate_sd3(
    prompt="A majestic snow-capped mountain reflected in a crystal-clear lake at sunrise, "
           "ultra-detailed, 8k resolution, dramatic lighting, photorealistic",
    negative_prompt="blurry, low quality, artifacts, deformed",
    cfg_scale=7.0,
    num_inference_steps=28,
    seed=42,
    device=DEVICE,
)

output.save("sd3_txt2img.png")
print(f"Saved: sd3_txt2img.png  (size={output.size})")
output

In [ ]:
!pip -q install -U diffusers transformers accelerate safetensors

import torch
from diffusers import AutoPipelineForText2Image
from IPython.display import display

pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=torch.float16,
    variant="fp16"
).to("cuda")
prompt="A majestic castle on a rocky cliff at sunset, dramatic golden light, 8k resolution, highly detailed, trending on artstation"
image = pipe(
    prompt,
    num_inference_steps=1,
    guidance_scale=0.0,
).images[0]

display(image)